<a href="https://colab.research.google.com/github/Hwk040319/MJY-ML/blob/main/01_test_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 배터리 열폭주 이미지 분류 · 제출용 실습

## 0. GPU 확인

In [1]:
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
# False 면 런타임 -> 런타임 유형 변경 -> T4 GPU 선택 후 이 셀 다시 실행

GPU 사용 가능: True


## 1. 코드 내려받기

In [2]:
# Google Drive에서 코드 ZIP을 가져옵니다.
# github에 있던 코드들의 ZIP파일이 반드시 구글 드라이브에 존재해야 합니다.
from google.colab import drive
from pathlib import Path
import zipfile


# Google Drive 연결
drive.mount("/content/drive")


# 코드 ZIP 위치(zip파일이 있는 경로로 수정!)
drive_folder = Path(
    "/content/drive/MyDrive/MJY"
)

code_zip_path = (
    drive_folder / "MJY-ML-main.zip"
)

project_dir = Path("/content/MJY-ML")


if not code_zip_path.is_file():
    raise FileNotFoundError(
        f"코드 ZIP을 찾을 수 없습니다: {code_zip_path}"
    )


# ZIP 안의 MJY-ML 폴더를 /content에 압축 해제합니다.
with zipfile.ZipFile(code_zip_path, "r") as archive:
    archive.extractall("/content")


if not (project_dir / "requirements.txt").is_file():
    raise FileNotFoundError(
        "ZIP 내부에 MJY-ML/requirements.txt가 없습니다."
    )


%cd /content/MJY-ML

!pip install -q -r requirements.txt

print("실습 코드 준비 완료")

Mounted at /content/drive
/content/MJY-ML
실습 코드 준비 완료


## 2. 데이터 내려받기



In [3]:
# 외부 Google Drive 공유 링크에서 데이터를 내려받습니다.

!pip install -q gdown


# 전체 Train / Validation 데이터
FILE_ID = "1PJNyDDdYd47wXD83DiW9PqFzbe7TLl0n"


# 데이터를 현재 프로젝트 폴더에 다운로드합니다.
# 데이터가 11GB이므로 다운받는데 시간이 걸립니다.
!gdown "https://drive.google.com/uc?id=$FILE_ID" \
    -O data.tar


# 압축 해제
!mkdir -p data

!tar -xf data.tar \
    -C data


# 데이터 폴더 확인
!ls data

Downloading...
From (original): https://drive.google.com/uc?id=1PJNyDDdYd47wXD83DiW9PqFzbe7TLl0n
From (redirected): https://drive.google.com/uc?id=1PJNyDDdYd47wXD83DiW9PqFzbe7TLl0n&confirm=t&uuid=938b080e-df02-4197-a9f2-cc2138699def
To: /content/MJY-ML/data.tar
100% 11.8G/11.8G [02:45<00:00, 71.4MB/s]
public_val  train


In [4]:
# 원하는 Train 비율을 선택합니다.
# Colab에서 슬라이더로 50%부터 90%까지 조절할 수 있습니다.
train_percent = 70  # @param {type:"slider", min:50, max:90, step:5}

from pathlib import Path
import shutil
import pandas as pd


# --------------------------------------------------
# 1. 데이터 경로 설정
# --------------------------------------------------

data_root = Path("data")

train_dir = data_root / "train"
val_dir = data_root / "public_val"

train_labels_path = train_dir / "labels.csv"
val_labels_path = val_dir / "labels.csv"

train_df = pd.read_csv(train_labels_path)
val_df = pd.read_csv(val_labels_path)


# --------------------------------------------------
# 2. 기존 데이터 누수 검사
# --------------------------------------------------

train_experiment_ids = set(
    train_df["experiment_id"].unique()
)

val_experiment_ids = set(
    val_df["experiment_id"].unique()
)

overlap = train_experiment_ids & val_experiment_ids

if overlap:
    raise RuntimeError(
        "Train과 Validation에 같은 experiment_id가 있습니다: "
        f"{sorted(overlap, key=str)}"
    )


# --------------------------------------------------
# 3. 목표 Train 실험 개수 계산
# --------------------------------------------------

total_experiments = (
    len(train_experiment_ids)
    + len(val_experiment_ids)
)

train_ratio = train_percent / 100

# 요청한 비율과 가장 가까운 실험 개수를 계산합니다.
desired_train_experiments = int(
    total_experiments * train_ratio + 0.5
)

# Train과 Validation에 최소 1개의 실험은 남도록 제한합니다.
desired_train_experiments = max(
    1,
    min(
        desired_train_experiments,
        total_experiments - 1
    )
)

current_train_experiments = len(
    train_experiment_ids
)

print(f"요청한 Train 비율: {train_percent}%")
print(f"전체 실험 수: {total_experiments}개")
print(
    f"목표 실험 수: "
    f"Train {desired_train_experiments}개 / "
    f"Valid {total_experiments - desired_train_experiments}개"
)


# --------------------------------------------------
# 4. experiment_id 단위로 이미지와 라벨 이동
# --------------------------------------------------

def move_experiments(
    source_df,
    destination_df,
    source_dir,
    destination_dir,
    experiment_ids
):
    """선택한 실험의 이미지와 라벨을 다른 분할로 이동합니다."""

    move_mask = source_df[
        "experiment_id"
    ].isin(experiment_ids)

    rows_to_move = source_df[
        move_mask
    ].copy()

    for image_name in rows_to_move["image_name"]:
        source_image = (
            source_dir / "images" / image_name
        )

        destination_image = (
            destination_dir / "images" / image_name
        )

        if not source_image.is_file():
            raise FileNotFoundError(
                f"이동할 이미지가 없습니다: {source_image}"
            )

        if destination_image.exists():
            raise FileExistsError(
                f"같은 파일이 이미 존재합니다: {destination_image}"
            )

        shutil.move(
            source_image,
            destination_image
        )

    # 출발 분할에서는 이동한 라벨을 제거합니다.
    source_df = source_df[
        ~move_mask
    ].reset_index(drop=True)

    # 도착 분할에는 이동한 라벨을 추가합니다.
    destination_df = pd.concat(
        [destination_df, rows_to_move],
        ignore_index=True
    )

    return source_df, destination_df


# Train 실험이 부족하면 Validation에서 가져옵니다.
if current_train_experiments < desired_train_experiments:
    number_to_move = (
        desired_train_experiments
        - current_train_experiments
    )

    move_ids = sorted(
        val_experiment_ids,
        key=str
    )[:number_to_move]

    val_df, train_df = move_experiments(
        source_df=val_df,
        destination_df=train_df,
        source_dir=val_dir,
        destination_dir=train_dir,
        experiment_ids=move_ids
    )

    print(
        "Validation에서 Train으로 이동한 실험:",
        move_ids
    )


# Train 실험이 너무 많으면 Validation으로 이동합니다.
elif current_train_experiments > desired_train_experiments:
    number_to_move = (
        current_train_experiments
        - desired_train_experiments
    )

    move_ids = sorted(
        train_experiment_ids,
        key=str
    )[:number_to_move]

    train_df, val_df = move_experiments(
        source_df=train_df,
        destination_df=val_df,
        source_dir=train_dir,
        destination_dir=val_dir,
        experiment_ids=move_ids
    )

    print(
        "Train에서 Validation으로 이동한 실험:",
        move_ids
    )


else:
    print("이미 요청한 비율에 맞는 실험 개수입니다.")


# --------------------------------------------------
# 5. 변경된 labels.csv 저장
# --------------------------------------------------

train_df.to_csv(
    train_labels_path,
    index=False
)

val_df.to_csv(
    val_labels_path,
    index=False
)


# --------------------------------------------------
# 6. 최종 결과 확인
# --------------------------------------------------

train_images = len(train_df)
val_images = len(val_df)
total_images = train_images + val_images

actual_train_percent = (
    train_images / total_images * 100
)

actual_val_percent = (
    val_images / total_images * 100
)

print()
print("-" * 55)

print(
    f"Train: {train_images}장 / "
    f"실험 {train_df['experiment_id'].nunique()}개 / "
    f"{actual_train_percent:.1f}%"
)

print(
    f"Valid: {val_images}장 / "
    f"실험 {val_df['experiment_id'].nunique()}개 / "
    f"{actual_val_percent:.1f}%"
)

print("-" * 55)

print("\nTrain 클래스 분포:")
print(
    train_df["target"]
    .value_counts()
    .sort_index()
)

print("\nValid 클래스 분포:")
print(
    val_df["target"]
    .value_counts()
    .sort_index()
)

요청한 Train 비율: 70%
전체 실험 수: 82개
목표 실험 수: Train 57개 / Valid 25개
Train에서 Validation으로 이동한 실험: [np.int64(20250805002), np.int64(20250806001), np.int64(20250806002), np.int64(20250806003), np.int64(20250808002), np.int64(20250808004), np.int64(20250811003), np.int64(20250811004), np.int64(20250811005), np.int64(20250813003)]

-------------------------------------------------------
Train: 133760장 / 실험 57개 / 72.3%
Valid: 51147장 / 실험 25개 / 27.7%
-------------------------------------------------------

Train 클래스 분포:
target
0    78219
1    42040
2    13501
Name: count, dtype: int64

Valid 클래스 분포:
target
0    26522
1    19702
2     4923
Name: count, dtype: int64


## 3. 데이터 검사




In [5]:
!python check_data.py --data-root data

데이터 검사 시작: /content/MJY-ML/data
------------------------------------------------------------
[train] 이미지 133760장 / 라벨 133760행
         초기  78219 ( 58.5%)  중기  42040 ( 31.4%)  후기  13501 ( 10.1%)
         experiment_id 57개
[public_val] 이미지 51147장 / 라벨 51147행
         초기  26522 ( 51.9%)  중기  19702 ( 38.5%)  후기   4923 (  9.6%)
         experiment_id 25개
[private_test] 폴더 없음 (정상 - 참가자에게 배포되지 않는 분할입니다)
------------------------------------------------------------
문제 없음. train_baseline.py 를 실행해도 좋습니다.


## 3.5 데이터 구조와 전처리 직접 확인 (선택)

채점이나 제출과 무관한 확인용 셀입니다. `train_baseline.py`를 실행하기 전에, 실제로 어떤 이미지가 어떤 라벨로 들어가는지, 그리고 5단계 중 Resize·Tensor 변환·정규화를 코드로 직접 확인합니다.

In [6]:
# labels.csv 구조 확인 — 이미지 파일명과 라벨이 어떻게 짝지어져 있는지
import pandas as pd

df = pd.read_csv('data/train/labels.csv')
print('열 구성:', list(df.columns))          # image_name, target, original_stage, experiment_id
print('행 개수:', len(df))
df.head()

열 구성: ['image_name', 'target', 'original_stage', 'experiment_id', 'source_partition', 'source_label_archive']
행 개수: 133760


,image_name,target,original_stage,experiment_id,source_partition,source_label_archive
0,20250813_140930_1.png,0,1,20250813004,Training,TL_센서_원통형_4.8_100_가열_20250813_20250813004.zip
1,20250813_140931_1.png,0,1,20250813004,Training,TL_센서_원통형_4.8_100_가열_20250813_20250813004.zip
2,20250813_140932_1.png,0,1,20250813004,Training,TL_센서_원통형_4.8_100_가열_20250813_20250813004.zip
3,20250813_140933_1.png,0,1,20250813004,Training,TL_센서_원통형_4.8_100_가열_20250813_20250813004.zip
4,20250813_140934_1.png,0,1,20250813004,Training,TL_센서_원통형_4.8_100_가열_20250813_20250813004.zip


In [7]:
# Colab 한글 폰트 설치 — 런타임마다 한 번 실행
!apt-get update -qq
!apt-get install -y -qq fonts-nanum > /dev/null

import matplotlib as mpl
import matplotlib.font_manager as fm
from pathlib import Path

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
assert Path(font_path).is_file(), f"폰트를 찾을 수 없습니다: {font_path}"

# 런타임을 재시작하지 않아도 현재 세션에 바로 등록
fm.fontManager.addfont(font_path)

mpl.rcParams["font.family"] = "NanumGothic"
mpl.rcParams["axes.unicode_minus"] = False

print("한글 폰트 설정 완료:", font_path)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
한글 폰트 설정 완료: /usr/share/fonts/truetype/nanum/NanumGothic.ttf


In [12]:
# 클래스(초기/중기/후기)별로 대표 이미지를 한 장씩 확인합니다.
# 모델 학습 전에 이미지와 라벨이 올바르게 연결되어 있는지
# 사람이 직접 눈으로 확인하기 위한 코드입니다.

from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd


# 이미지 파일명, 분류 라벨, 원본 단계 등이 저장된 CSV를 불러옵니다.
df = pd.read_csv("data/train/labels.csv")

# target 숫자가 의미하는 클래스 이름입니다.
# target=0: 초기, target=1: 중기, target=2: 후기
CLASS_NAMES = ["초기", "중기", "후기"]


# 초기·중기·후기 이미지를 가로로 한 장씩 표시할 공간을 만듭니다.
# 1행 3열이므로 총 3개의 그래프 영역이 만들어집니다.
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))


# cls에 0, 1, 2가 순서대로 들어갑니다.
# 따라서 이미지도 초기 → 중기 → 후기 순서로 표시됩니다.
for cls in range(3):

    # 현재 클래스에 해당하는 행들 중 대표 샘플을 선택합니다.
    row = df[df["target"] == cls].iloc[100]

    # labels.csv에 기록된 이미지 파일명을 이용해 실제 이미지를 불러옵니다.
    # convert("RGB")는 이미지를 빨강·초록·파랑의 3채널 형식으로 통일합니다.
    image_path = f"data/train/images/{row['image_name']}"
    img = Image.open(image_path).convert("RGB")

    # 현재 클래스에 해당하는 그래프 영역에 이미지를 표시합니다.
    axes[cls].imshow(img)

    # 이미지 위에 클래스 이름, target 번호, 원본 단계를 표시합니다.
    # 파일명은 화면이 복잡해지지 않도록 표시하지 않습니다.
    axes[cls].set_title(
        f"{CLASS_NAMES[cls]}\n"
        f"(target={row['target']}, 원본 단계={row['original_stage']})",
        fontsize=12,
        pad=8
    )

    # 이미지 주변의 좌표축과 눈금은 필요하지 않으므로 숨깁니다.
    axes[cls].axis("off")


# 세 이미지 전체를 설명하는 제목을 그림 위쪽에 표시합니다.
fig.suptitle(
    "클래스별 샘플 이미지 — 초기 · 중기 · 후기",
    fontsize=16,
    y=0.98
)


# 전체 제목과 각 이미지 제목이 서로 겹치지 않도록 여백을 조정합니다.
fig.subplots_adjust(
    left=0.03,     # 그림 왼쪽 여백
    right=0.97,    # 그림 오른쪽 여백
    bottom=0.03,   # 그림 아래쪽 여백
    top=0.76,      # 전체 제목을 위한 위쪽 공간
    wspace=0.15    # 이미지 사이의 가로 간격
)


# 완성된 결과를 화면에 출력합니다.
plt.show()

<Figure size 1300x450 with 3 Axes>

In [10]:
# 초기·중기·후기에서 선택된 이미지 정보를 각각 확인합니다.

for cls in range(3):
    row = df[df["target"] == cls].iloc[0]

    print(
        f"{CLASS_NAMES[cls]}: "
        f"파일={row['image_name']}, "
        f"실험={row['experiment_id']}, "
        f"원본 단계={row['original_stage']}"
    )

초기: 파일=20250813_140930_1.png, 실험=20250813004, 원본 단계=1
중기: 파일=20250813_141716_1.png, 실험=20250813004, 원본 단계=3
후기: 파일=20250813_142720_1.png, 실험=20250813004, 원본 단계=5


In [14]:
# 전처리 흐름(Resize -> ToTensor -> Normalize) 실행
from torchvision import transforms
from common import IMAGENET_MEAN, IMAGENET_STD, get_transforms

sample_name = df.iloc[0]['image_name']
sample_img = Image.open(f"data/train/images/{sample_name}").convert('RGB')
print('0) 원본 크기:', sample_img.size)

step1 = transforms.Resize((224, 224))(sample_img)
print('1) Resize 후 크기:', step1.size)                      # (224, 224)로 통일

step2 = transforms.ToTensor()(step1)
print('2) ToTensor 후 shape:', tuple(step2.shape),
      '값 범위:', round(step2.min().item(), 3), '~', round(step2.max().item(), 3))
# [3, 224, 224], 0~255 픽셀값을 255로 나눠 0~1 범위로 변환

step3 = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)(step2)
print('3) Normalize 후 값 범위:', round(step3.min().item(), 2), '~', round(step3.max().item(), 2))
# ImageNet 평균/표준편차로 재조정 -> 사전학습된 ResNet18이 기대하는 입력 분포에 맞춤

final = get_transforms(224, train=False)(sample_img)
print('get_transforms() 결과와 동일한가:', torch.allclose(final, step3))  # True면 위 세 단계와 같은 전처리

0) 원본 크기: (45, 76)
1) Resize 후 크기: (224, 224)
2) ToTensor 후 shape: (3, 224, 224) 값 범위: 0.0 ~ 0.988
3) Normalize 후 값 범위: -2.12 ~ 2.59
get_transforms() 결과와 동일한가: True


## 4. 모델 학습

In [19]:
# 가장 최적의 결과를 나타내는 파라미터 찾기
# 아래는 하나의 예시
# 현재 crop-scale-min, flip-prob, rotation-degrees, brightness, contrast만 사용 가능. 추가 파라미터 원할 시 코드 ZIP 파일에 있는 common.py 수정
# 현재 adam, adamw, sgd 3개 사용 가능. optimizer 추가 시 코드 ZIP 파일에 있는 train_baseline.py 수정
!python train_baseline.py \
    --data-root data \
    --augment \
    --crop-scale-min 0.75 \
    --flip-prob 0.5 \
    --rotation-degrees 15 \
    --brightness 0.3 \
    --contrast 0.3 \
    --epochs 5 \
    --optimizer adamw \
    --output-dir outputs/test

usage: train_baseline.py [-h] [--data-root DATA_ROOT]
                         [--output-dir OUTPUT_DIR] [--epochs EPOCHS]
                         [--batch-size BATCH_SIZE] [--lr LR]
                         [--weight-decay WEIGHT_DECAY]
                         [--image-size IMAGE_SIZE] [--seed SEED]
                         [--num-workers NUM_WORKERS] [--augment]
                         [--use-class-weights] [--unfreeze]
                         [--optimizer {adamw,adam,sgd}]
train_baseline.py: error: unrecognized arguments: --crop-scale-min 0.75 --flip-prob 0.5 --rotation-degrees 15 --brightness 0.3 --contrast 0.3


## 5. 결과 확인

## 이미지 한 장 예측 (선택)

In [21]:
from pathlib import Path
from PIL import Image
from IPython.display import display

# public_val 이미지 중 한 장을 선택합니다.
sample_image = next(Path("data/public_val/images").glob("*.png"))

# 선택한 이미지를 화면에 표시합니다.
display(Image.open(sample_image))

# 표시된 이미지를 학습된 모델로 예측합니다.
!python predict_one.py \
    --image "$sample_image" \
    --checkpoint outputs/test/best_model.pt

<PIL.PngImagePlugin.PngImageFile image mode=RGB size=40x76>

Traceback (most recent call last):
  File "/content/MJY-ML/predict_one.py", line 62, in <module>
    main()
    ~~~~^^
  File "/content/MJY-ML/predict_one.py", line 30, in main
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 1530, in load
    with _open_file_like(f, "rb") as opened_file:
         ~~~~~~~~~~~~~~~^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 795, in _open_file_like
    return _open_file(name_or_buffer, mode)
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 776, in __init__
    super().__init__(open(name, mode))  # noqa: SIM115
                     ~~~~^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'outputs/test/best_model.pt'


## 학습 지표 확인

In [ ]:
# 이 셀만 실행해도 그래프가 나오도록 필요한 결과를 먼저 불러옵니다.
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage

# 학습할 때 --output-dir로 지정한 폴더와 같아야 합니다.
output_dir = Path("outputs/test")
report_path = output_dir / "validation_report.json"
learning_curves_path = output_dir / "learning_curves.png"
confusion_matrix_path = output_dir / "confusion_matrix.png"

# 학습이 완료되지 않았거나 output_dir이 다르면 이해하기 쉬운 오류를 표시합니다.
for result_path in [report_path, learning_curves_path, confusion_matrix_path]:
    if not result_path.is_file():
        raise FileNotFoundError(
            f"결과 파일을 찾을 수 없습니다: {result_path}\n"
            "먼저 모델 학습 셀을 실행하고 output_dir 경로를 확인하세요."
        )

with open(report_path, encoding="utf-8") as file:
    report = json.load(file)

learning_curves = np.asarray(PILImage.open(learning_curves_path).convert("RGB"))
confusion_matrix_image = np.asarray(
    PILImage.open(confusion_matrix_path).convert("RGB")
)

# 혼동행렬의 각 행은 실제 클래스, 각 열은 예측 클래스를 뜻합니다.
CLASS_NAMES = ["초기", "중기", "후기"]
cm = np.asarray(report["confusion_matrix"], dtype=float)
row_totals = cm.sum(axis=1)
class_recall = np.divide(
    np.diag(cm),
    row_totals,
    out=np.zeros_like(row_totals),
    where=row_totals != 0,
)
macro_recall = float(class_recall.mean())

print("Accuracy:", round(report["accuracy"], 4))
print("Macro F1:", round(report["macro_f1"], 4))
print("Macro Recall:", round(macro_recall, 4))

# 세 그래프를 작게 만들어 가로로 나란히 표시합니다.
fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 4),
    gridspec_kw={
        "width_ratios": [1.7, 1, 1]
    }
)


# 1. 학습 곡선
axes[0].imshow(learning_curves)
axes[0].axis("off")


# 2. Confusion Matrix
axes[1].imshow(confusion_matrix_image)
axes[1].axis("off")


# 3. 클래스별 Recall
recall_bars = axes[2].bar(
    CLASS_NAMES,
    class_recall,
    color=[
        "#5B8FF9",
        "#61DDAA",
        "#F6BD16"
    ]
)

# 막대 위에 Recall 값을 표시합니다.
axes[2].bar_label(
    recall_bars,
    labels=[
        f"{value:.3f}"
        for value in class_recall
    ],
    padding=3
)

axes[2].set_ylim(0, 1.08)
axes[2].set_xlabel("배터리 상태")
axes[2].set_ylabel("Recall")
axes[2].set_title(
    f"클래스별 Recall\nMacro Recall: {macro_recall:.4f}"
)
axes[2].grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()
plt.show()

<Figure size 1500x400 with 3 Axes>

## 6. 최종 제출 · 2회차 전날 23:59 마감

In [ ]:
import torch
FINAL = 'outputs/test/best_model.pt'   # 최종 선택한 경로로 변경
ckpt = torch.load(FINAL, map_location='cpu', weights_only=False)
print('Public Val Macro F1:', round(ckpt['macro_f1'], 4), '| epoch:', ckpt['epoch'])

from google.colab import files
files.download(FINAL)
# [팀명]_best_model.pt 로 이름 변경 후 이메일로 제출